# DualBranch CellPose v10 — 推理 Notebook v2
## 直接读取 train/val/test 文件夹 · 修复聚集体判断 · 修复全部 P0-P2 Bug

### 修复清单
| 编号 | 问题 | 修复 |
|------|------|------|
| P0 | 形态学参数过激（opening=2/dil=8/ero=8）导致实例碎裂或合并 | opening=1 / dilation=5 / erosion=5 |
| P1-a | BSIZE=512 与训练不一致，pos_embed 插值失真 | BSIZE=256 |
| P1-b | PROB_THRESHOLD=0.4 低于 BCE 自然边界 | PROB_THRESHOLD=0.5 |
| P2-a | 明场图未转 (H,W,3) 导致 sliding_window pad 出错 | bf_to_3ch() 强制转换 |
| P2-b | CSV 实例行重复写入 | drop_duplicates 去重 |
| **AGG** | **`is_aggregate` 全为 1（最核心问题）** | **见下方详细说明** |

### `is_aggregate` 全为 1 的根因（三重叠加）
1. **`NOISE_AREA_PX=1000`（6.4 µm²）过大**：把 <6.4 µm² 的连通域全部丢弃，残留的实例面积偏大，几乎全部超过面积阈值。修复：`NOISE_AREA_PX=300`（过滤真噪点 <1.9 µm²，保留正常小血小板）
2. **`AREA_THRESHOLD_UM2=48 µm²` 过小**：静息血小板面积约 20–80 µm²，48 µm² 阈值把大量正常血小板误判为聚集体。修复：`AREA_THRESHOLD_UM2=150 µm²`（约等效直径 14 µm，明确区分单体与簇）
3. **`SOLIDITY_THRESHOLD=0.85` 过严**：形态学后处理轻微改变轮廓后许多正常血小板的 solidity 略低于 0.85。修复：`SOLIDITY_THRESHOLD=0.75`

### 目录结构约定
```
DATA_ROOT/
  train/
    220211 platelet cangrelor 10 ugml sten no agonist new/
      10min/images_v15_only colorbar_v2/
        img_001_phase.tif
        img_001_intensity.tif   ← 或 brightfield
      ...
  val/
    （相同结构）
  test/
    （相同结构）
```

In [15]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §0  安装依赖                                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess
subprocess.run(["pip", "install", "-q", "git+https://github.com/mouseland/cellpose.git"], check=True)
subprocess.run(["pip", "install", "-q", "segmentation_models_pytorch", "six", "pandas",
                "tifffile", "scikit-image", "scikit-learn", "tqdm"], check=True)
print('依赖安装完成 ✓')

依赖安装完成 ✓


In [16]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §1  挂载 Google Drive                                               ║
# ╚══════════════════════════════════════════════════════════════════════╝
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

Mounted at /content/drive


# ║  §2  全局配置  ← 每次只需修改本节                                   ║


In [31]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
import os, re
import numpy as np

# ── 路径 ──────────────────────────────────────────────────────────────
BEST_CKPT_PATH = '/content/drive/MyDrive/drug_effect/seg/ckpts/20260326_103536/best.pth'

# train/val/test 三个子目录所在的根目录
DATA_ROOT   = '/content/drive/MyDrive/drug_effect/'
OUTPUT_DIR  = '/content/drive/MyDrive/drug_effect/cellpose'
CSV_OUT_DIR = os.path.join(OUTPUT_DIR, 'features')
MASK_OUT_DIR = os.path.join(OUTPUT_DIR, 'masks')
os.makedirs(CSV_OUT_DIR,  exist_ok=True)
os.makedirs(MASK_OUT_DIR, exist_ok=True)

# 定义备份 CSV 文件路径，使其在其他单元格中可用
BACKUP_CSV = os.path.join(CSV_OUT_DIR, 'BACKUP_latest.csv')

# ── 文件命名关键字 ────────────────────────────────────────────────────
# 相位图文件名必须含 PHASE_KW；明场图含 INTENSITY_KW（顺序查找）
PHASE_KW     = '_phase'                          # 例：img001_phase.tif
INTENSITY_KW = ['_intensity', '_brightfield', '_bf']  # 任一匹配即可
IMAGE_EXTS   = {'.tiff', '.tif', '.png', '.jpg', '.jpeg'}

# ── 物理参数 ──────────────────────────────────────────────────────────
PIXEL_TO_MICRON = 0.08   # µm/pixel
PIXEL_SIZE      = PIXEL_TO_MICRON
WAVELENGTH      = 1030e-9
ALPHA           = 0.2

# ── 推理参数（P0/P1 修复）────────────────────────────────────────────
BSIZE           = 256    # ★ P1修复：与训练一致（原加速版错误设为512） -- 修复为256以匹配检查点中32x32的pos_embed
SW_STRIDE_RATIO = 0.25
PROB_THRESHOLD  = 0.5    # ★ P1修复：BCE自然阈值（原加速版错误设为0.4）
INFER_BATCH     = 128     # tile批量，显存不足时调小
SAVE_MASKS      = True

# ── 形态学参数（P0 修复）─────────────────────────────────────────────
MORPH_OPENING  = 1       # ★ P0修复：原加速版错误使用2
MORPH_DILATION = 5       # ★ P0修复：原加速版错误使用8
MORPH_EROSION  = 5       # ★ P0修复：原加速版错误使用8

# ── 噪声过滤（★ AGG修复1）────────────────────────────────────────────
# 原值 1000 px (6.4 µm²) 过大，过滤掉了正常小血小板，导致残留实例面积偏大
# 修复为 300 px (1.9 µm²)，仅过滤真实噪点
NOISE_AREA_PX = 600

# ── 聚集体判定阈值（★ AGG修复2/3）───────────────────────────────────
#
# 血小板物理尺寸参考（0.08 µm/px）：
#   静息血小板：直径 2-5 µm，面积 3-20 µm²（469-3125 px）
#   激活血小板：直径 5-12 µm，面积 20-113 µm²（3125-17671 px）
#   两个激活血小板接触：面积 ~40-226 µm²
#   明确聚集体（3个+）：面积 >150 µm²（>23438 px）
#
# ★ AGG修复2：面积阈值从 48 µm² 提高到 150 µm²
#   原值 48 µm² 等效直径仅 7.8 µm，把大量激活单体血小板误判为聚集体
AREA_THRESHOLD_UM2 = 45   # µm²

# ★ AGG修复3：solidity 阈值从 0.85 放宽到 0.75
#   原值过严，形态学处理后轮廓轻微不规则即被误判
SOLIDITY_THRESHOLD     = 0.75
ASPECT_RATIO_THRESHOLD = 2.0  # 长宽比阈值也适当放宽（原值1.8过严）

# ── 边缘过滤 ─────────────────────────────────────────────────────────
EDGE_MARGIN_PX = 30      # 质心距图像边缘 < 此像素的实例丢弃（适当收紧）
MIN_INSTANCE_PX = 5

# ── 邻居密度判定 ──────────────────────────────────────────────────────
NEIGHBOR_DIST_THR  = 90  # 质心距离 < 90 px 视为邻居
NEIGHBOR_COUNT_THR = 2   # 邻居数 > 3 才标记为聚集体（原值2过松）


# ── 并发参数 ──────────────────────────────────────────────────────────
FEAT_WORKERS  = 8
SAVE_INTERVAL = 50   # 每处理 N 张图备份一次

# ── 目录映射表（来自题目）────────────────────────────────────────────
# key: 相对于 split 根目录的路径，value: folder_id（备用）
FOLDER_MAP = {
    "220211 platelet cangrelor 10 ugml sten no agonist new/10min/images_v15_only colorbar_v2": "01",
    "220211 platelet cangrelor 10 ugml sten no agonist new/15min/images_v15_only colorbar_v2": "02",
    "220211 platelet cangrelor 10 ugml sten no agonist new/20min/images_v15_only colorbar_v2": "03",
    "220211 platelet cangrelor 10 ugml sten no agonist new/5min/images_v15_only colorbar_v2":  "04",
    "220211 platelet cangrelor 10 ugml sten no agonist new/control/images_v15_only colorbar_v2": "05",
    "220215 platelet aspirin 0.25 mM no agonist/10min/images_v15_only colorbar_v2": "06",
    "220215 platelet aspirin 0.25 mM no agonist/15min/images_v15_only colorbar_v2": "07",
    "220215 platelet aspirin 0.25 mM no agonist/20min/images_v15_only colorbar_v2": "08",
    "220215 platelet aspirin 0.25 mM no agonist/5min/images_v15_only colorbar_v2":  "09",
    "220215 platelet aspirin 0.25 mM no agonist/control/images_v15_only colorbar_v2": "10",
    "220226 platelet no durg no agonist new/10min/images_v15_only colorbar_v3": "11",
    "220226 platelet no durg no agonist new/15min/images_v15_only colorbar_v3": "12",
    "220226 platelet no durg no agonist new/20min/images_v15_only colorbar_v3": "13",
    "220226 platelet no durg no agonist new/5min/images_v15_only colorbar_v3":  "14",
    "220226 platelet no durg no agonist new/control/images_v15_only colorbar_v3": "15",
    "220310 platelet eptifibatide no agonist new/10min/images_v15_only colorbar_v3": "16",
    "220310 platelet eptifibatide no agonist new/15min/images_v15_only colorbar_v3": "17",
    "220310 platelet eptifibatide no agonist new/20min/images_v15_only colorbar_v3": "18",
    "220310 platelet eptifibatide no agonist new/5min/images_v15_only colorbar_v3":  "19",
    "220310 platelet eptifibatide no agonist new/control/images_v15_only colorbar_v3": "20",
}

# 从路径推断 drug / time_pt 的正则
_DRUG_RE = re.compile(
    r'(?P<drug>cangrelor|aspirin|eptifibatide|no.?d[ur]+g)',
    re.IGNORECASE
)
_TIME_RE = re.compile(
    r'(?P<time>\d+\s*min|control)',
    re.IGNORECASE
)

def _parse_meta(rel_path: str):
    """从相对路径字符串提取 drug / time_pt / folder_id。"""
    # 规范化路径分隔符
    rel_norm = rel_path.replace('\\', '/')

    # drug
    m_drug = _DRUG_RE.search(rel_norm)
    if m_drug:
        raw = m_drug.group('drug').lower()
        if 'no' in raw and ('drug' in raw or 'durg' in raw):
            drug = 'nodrug'
        elif 'cangrelor' in raw:
            drug = 'cangrelor'
        elif 'aspirin' in raw:
            drug = 'aspirin'
        elif 'eptifibatide' in raw:
            drug = 'eptifibatide'
        else:
            drug = 'unknown'
    else:
        drug = 'unknown'

    # time_pt
    m_time = _TIME_RE.search(rel_norm)
    if m_time:
        raw_t = m_time.group('time').lower().replace(' ', '')
        time_pt = '0min' if raw_t == 'control' else raw_t
    else:
        time_pt = 'unknown'

    # folder_id
    folder_id = ''
    for k, v in FOLDER_MAP.items():
        if k in rel_norm:
            folder_id = v
            break

    return drug, time_pt, folder_id


print('§2 配置完成 ✓')
print(f'  BSIZE={BSIZE}  PROB_THRESHOLD={PROB_THRESHOLD}')
print(f'  形态学: opening={MORPH_OPENING} dilation={MORPH_DILATION} erosion={MORPH_EROSION}')
print(f'  NOISE_AREA_PX={NOISE_AREA_PX} ({NOISE_AREA_PX*PIXEL_TO_MICRON**2:.1f} µm²)')
print(f'  AREA_THRESHOLD_UM2={AREA_THRESHOLD_UM2} µm²  SOLIDITY={SOLIDITY_THRESHOLD}  ASPECT={ASPECT_RATIO_THRESHOLD}')

§2 配置完成 ✓
  BSIZE=256  PROB_THRESHOLD=0.5
  形态学: opening=1 dilation=5 erosion=5
  NOISE_AREA_PX=600 (3.8 µm²)
  AREA_THRESHOLD_UM2=45 µm²  SOLIDITY=0.75  ASPECT=2.0


# ║  §3  图像扫描（遍历 train/val/test 三个 split）                     ║


In [18]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
from pathlib import Path
from typing import List, Dict, Optional, Tuple


def _find_intensity_path(phase_path: Path) -> Optional[Path]:
    """在同目录下找对应的明场图，支持多个候选关键字。"""
    for kw in INTENSITY_KW:
        int_stem = phase_path.stem.replace(PHASE_KW, kw)
        # 先尝试同扩展名
        cand = phase_path.parent / (int_stem + phase_path.suffix)
        if cand.exists():
            return cand
        # 再尝试其他扩展名
        for ext in IMAGE_EXTS:
            cand = phase_path.parent / (int_stem + ext)
            if cand.exists():
                return cand
    return None


def scan_split(split_name: str) -> List[Dict]:
    """
    扫描 DATA_ROOT/{split_name}/ 下所有相位图，配对明场图。
    返回样本列表，每条记录包含：
      phase / intensity / image_name / stem / split / rel_dir /
      drug / time_pt / folder_id
    """
    split_root = Path(DATA_ROOT) / split_name
    if not split_root.exists():
        print(f'  [SKIP] {split_root} 不存在')
        return []

    samples = []
    skipped = []

    for phase_path in sorted(split_root.rglob('*')):
        if not phase_path.is_file():
            continue
        if phase_path.suffix.lower() not in IMAGE_EXTS:
            continue
        if PHASE_KW not in phase_path.stem:
            continue

        int_path = _find_intensity_path(phase_path)
        if int_path is None:
            skipped.append(phase_path.name)
            continue

        # 相对路径（用于元信息解析和输出目录组织）
        rel_dir = str(phase_path.parent.relative_to(split_root))
        drug, time_pt, folder_id = _parse_meta(rel_dir)

        samples.append({
            'phase'     : str(phase_path),
            'intensity' : str(int_path),
            'image_name': phase_path.name,
            'stem'      : phase_path.stem.replace(PHASE_KW, '').strip('_'),
            'split'     : split_name,
            'rel_dir'   : rel_dir,
            'folder_id' : folder_id,
            'drug'      : drug,
            'time_pt'   : time_pt,
            'unique_key': f"{split_name}/{rel_dir}/{phase_path.name}",
        })
    print(f'  [{split_name}] 找到 {len(samples)} 对图像'
          + (f'，跳过 {len(skipped)} 张（找不到明场图）' if skipped else ''))

    # 药物/时间点分布统计
    from collections import Counter
    dist = Counter(f"{s['drug']}/{s['time_pt']}" for s in samples)
    for k, v in sorted(dist.items()):
        print(f'    {k}: {v} 张')

    return samples


# ── 扫描三个 split ────────────────────────────────────────────────────
print('开始扫描图像...')
all_samples: List[Dict] = []
for _split in ['test', 'val', 'train']:
    all_samples.extend(scan_split(_split))

print(f'\n合计: {len(all_samples)} 对图像')
if not all_samples:
    raise RuntimeError(f'未扫描到任何图像，请检查 DATA_ROOT={DATA_ROOT} 和 PHASE_KW={PHASE_KW}')

开始扫描图像...
  [test] 找到 9031 对图像
    aspirin/0min: 342 张
    aspirin/10min: 426 张
    aspirin/15min: 450 张
    aspirin/20min: 419 张
    aspirin/5min: 374 张
    cangrelor/0min: 439 张
    cangrelor/10min: 440 张
    cangrelor/15min: 462 张
    cangrelor/20min: 468 张
    cangrelor/5min: 449 张
    eptifibatide/0min: 496 张
    eptifibatide/10min: 496 张
    eptifibatide/15min: 495 张
    eptifibatide/20min: 496 张
    eptifibatide/5min: 497 张
    nodrug/0min: 469 张
    nodrug/10min: 448 张
    nodrug/15min: 455 张
    nodrug/20min: 455 张
    nodrug/5min: 455 张
  [val] 找到 9074 对图像
    aspirin/0min: 358 张
    aspirin/10min: 411 张
    aspirin/15min: 448 张
    aspirin/20min: 419 张
    aspirin/5min: 407 张
    cangrelor/0min: 452 张
    cangrelor/10min: 434 张
    cangrelor/15min: 473 张
    cangrelor/20min: 458 张
    cangrelor/5min: 464 张
    eptifibatide/0min: 497 张
    eptifibatide/10min: 496 张
    eptifibatide/15min: 493 张
    eptifibatide/20min: 498 张
    eptifibatide/5min: 495 张
    nodrug/0min: 462 张


# ║  §4  图像读取工具                                                    ║


In [19]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
import cv2
import tifffile


def read_image_rgb(path: str) -> np.ndarray:
    """读取任意格式图像 → (H,W,3) uint8 RGB。"""
    ext = Path(path).suffix.lower()
    if ext in ('.tiff', '.tif'):
        img = tifffile.imread(path)
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        elif img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = img.transpose(1, 2, 0)
        if img.ndim == 3 and img.shape[-1] == 4:
            img = img[..., :3]
        if img.ndim == 3 and img.shape[-1] == 1:
            img = np.concatenate([img] * 3, axis=-1)
    else:
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f'无法读取: {path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # 归一化到 uint8
    if img.dtype != np.uint8:
        vmax = img.max()
        img = (img.astype(np.float32) / vmax * 255).astype(np.uint8) if vmax > 0 else img.astype(np.uint8)
    return img


def read_image_gray(path: str) -> np.ndarray:
    """读取明场图 → (H,W) float32 [0,1]。"""
    return read_image_rgb(path).mean(axis=-1).astype(np.float32) / 255.0


def bf_to_3ch(bf_gray: np.ndarray) -> np.ndarray:
    """(H,W) float32 → (H,W,3) uint8。★ P2修复：滑窗推理要求三通道输入。"""
    u8 = (bf_gray * 255).clip(0, 255).astype(np.uint8)
    return np.stack([u8, u8, u8], axis=-1)


print('§4 图像读取工具就绪 ✓')

§4 图像读取工具就绪 ✓


# ║  §5  模型定义 & 推理工具                                             ║


In [20]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
from cellpose import transforms as cp_tf
from cellpose.vit_sam import Transformer
from scipy import ndimage


def normalize_cp(img_hwc: np.ndarray) -> np.ndarray:
    return cp_tf.normalize_img(img_hwc.astype(np.float32))


class FusionBlock(nn.Module):
    def __init__(self, channels: int = 256):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
            nn.GroupNorm(32, channels),
            nn.GELU(),
        )
        nn.init.kaiming_normal_(self.fuse[0].weight, mode='fan_out', nonlinearity='relu')

    def forward(self, fp, fi):
        return self.fuse(torch.cat([fp, fi], dim=1))


class DualBranchTransformer(Transformer):
    def __init__(self, pretrained_path=None, backbone='vit_l',
                 ps=8, nout=1, bsize=256, rdrop=0.4, dtype=torch.float32):
        super().__init__(backbone=backbone, ps=ps, nout=nout,
                         bsize=bsize, rdrop=rdrop, checkpoint=None, dtype=dtype)
        self.fusion = FusionBlock(channels=256)
        if pretrained_path and os.path.exists(pretrained_path):
            self._load_weights(pretrained_path)

    def _load_weights(self, path: str):
        raw = torch.load(path, map_location='cpu', weights_only=False)
        sd = raw['model'] if isinstance(raw, dict) and 'model' in raw else \
             raw['state_dict'] if isinstance(raw, dict) and 'state_dict' in raw else raw
        if isinstance(sd, dict) and any(k.startswith('module.') for k in sd):
            sd = OrderedDict({k[7:]: v for k, v in sd.items()})
        own_sd = self.state_dict()
        filtered = OrderedDict()
        for k, v in sd.items():
            if k in {'out.weight', 'out.bias', 'W2'} and k in own_sd and own_sd[k].shape != v.shape:
                continue
            filtered[k] = v
        self.load_state_dict(filtered, strict=False)

    def encode(self, x):
        x = self.encoder.patch_embed(x)
        if self.encoder.pos_embed is not None:
            pe = self.encoder.pos_embed
            if pe.shape[1:3] != x.shape[1:3]:
                pe = F.interpolate(pe.permute(0,3,1,2).float(),
                                   size=x.shape[1:3], mode='bilinear',
                                   align_corners=False).to(x.dtype).permute(0,2,3,1)
            x = x + pe
        for blk in self.encoder.blocks:
            x = blk(x)
        return self.encoder.neck(x.permute(0,3,1,2))

    def decode(self, feat):
        return F.conv_transpose2d(self.out(feat), self.W2, stride=self.ps, padding=0)

    def forward(self, phase, intensity):
        fused = self.fusion(self.encode(phase), self.encode(intensity))
        out   = self.decode(fused)
        return out, torch.zeros((phase.shape[0], 256), device=phase.device, dtype=out.dtype)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -80, 80)))


def pred_to_mask(pred_np: np.ndarray) -> np.ndarray:
    """
    [1,H,W] logit → [H,W] uint32 实例 mask。
    ★ P0修复：形态学参数从配置读取（opening=1, dil=5, ero=5）
    """
    H, W     = pred_np.shape[1], pred_np.shape[2]
    cellprob = sigmoid(pred_np[0]).reshape(H, W).astype(np.float32)
    binary   = (cellprob > PROB_THRESHOLD).astype(np.uint8)
    binary   = ndimage.binary_opening(binary,  iterations=MORPH_OPENING ).astype(np.uint8)
    binary   = ndimage.binary_fill_holes(binary).astype(np.uint8)
    binary   = ndimage.binary_dilation(binary, iterations=MORPH_DILATION).astype(np.uint8)
    binary   = ndimage.binary_erosion(binary,  iterations=MORPH_EROSION ).astype(np.uint8)
    labeled, _ = ndimage.label(binary)
    # ★ AGG修复1：噪声过滤阈值从1000降至300
    if NOISE_AREA_PX > 0 and labeled.max() > 0:
        counts      = np.bincount(labeled.ravel())
        counts[0]   = NOISE_AREA_PX + 1
        small_ids   = np.where(counts < NOISE_AREA_PX)[0]
        if small_ids.size > 0:
            labeled[np.isin(labeled, small_ids)] = 0
        labeled, _ = ndimage.label(labeled > 0)
    return labeled.astype(np.uint32)


_gauss_cache: dict = {}

def _get_gauss_w(tile_size: int) -> np.ndarray:
    if tile_size not in _gauss_cache:
        sigma = tile_size / 8.0
        cy, cx = np.mgrid[0:tile_size, 0:tile_size]
        _gauss_cache[tile_size] = np.exp(
            -((cy - tile_size/2)**2 + (cx - tile_size/2)**2) / (2*sigma**2)
        ).astype(np.float32)
    return _gauss_cache[tile_size]


def sliding_window_inference(eval_model, ph_arr, it_arr,
                              tile_size=None, stride_ratio=None,
                              device=None, infer_batch=None):
    """高斯加权滑窗推理。★ P2修复：it_arr 必须是 (H,W,3)。"""
    if tile_size    is None: tile_size    = BSIZE
    if stride_ratio is None: stride_ratio = SW_STRIDE_RATIO
    if device       is None: device       = next(eval_model.parameters()).device
    if infer_batch  is None: infer_batch  = INFER_BATCH

    H_orig, W_orig = ph_arr.shape[:2]
    stride = max(1, int(tile_size * (1 - stride_ratio)))
    pad_h  = max(tile_size - H_orig, 0)
    pad_w  = max(tile_size - W_orig, 0)
    if H_orig > tile_size: pad_h += (stride - (H_orig - tile_size) % stride) % stride
    if W_orig > tile_size: pad_w += (stride - (W_orig - tile_size) % stride) % stride
    ph_pad = np.pad(ph_arr, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    it_pad = np.pad(it_arr, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    H_pad, W_pad = ph_pad.shape[:2]

    gauss_w    = _get_gauss_w(tile_size)
    logit_acc  = np.zeros((H_pad, W_pad), dtype=np.float32)
    weight_acc = np.zeros((H_pad, W_pad), dtype=np.float32)
    ys = list(range(0, H_pad - tile_size + 1, stride))
    xs = list(range(0, W_pad - tile_size + 1, stride))

    tiles_ph, tiles_it, coords = [], [], []
    for y0 in ys:
        for x0 in xs:
            tiles_ph.append(normalize_cp(ph_pad[y0:y0+tile_size, x0:x0+tile_size]).transpose(2,0,1))
            tiles_it.append(normalize_cp(it_pad[y0:y0+tile_size, x0:x0+tile_size]).transpose(2,0,1))
            coords.append((y0, x0))

    use_amp = (device.type == 'cuda')
    eval_model.eval()
    with torch.no_grad():
        for b in range(0, len(tiles_ph), infer_batch):
            ph_b = torch.from_numpy(np.stack(tiles_ph[b:b+infer_batch])).to(device, non_blocking=True)
            it_b = torch.from_numpy(np.stack(tiles_it[b:b+infer_batch])).to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out, _ = eval_model(ph_b, it_b)
            out_np = out.float().cpu().numpy()
            for j, (y0, x0) in enumerate(coords[b:b+infer_batch]):
                logit_acc [y0:y0+tile_size, x0:x0+tile_size] += out_np[j, 0] * gauss_w
                weight_acc[y0:y0+tile_size, x0:x0+tile_size] += gauss_w

    logit_full = (logit_acc / np.maximum(weight_acc, 1e-8))[:H_orig, :W_orig]
    return logit_full[np.newaxis].astype(np.float32), ph_arr, it_arr


print('§5 模型与推理工具定义完成 ✓')

§5 模型与推理工具定义完成 ✓


# ║  §6  特征提取                                                        ║


In [21]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
import re as _re
from scipy import stats
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import regionprops


def color_to_phase(phase_rgb: np.ndarray, phase_range: float = 2*np.pi) -> np.ndarray:
    hsv = cv2.cvtColor(phase_rgb.astype(np.uint8), cv2.COLOR_RGB2HSV)
    return hsv[..., 0].astype(np.float32) / 180.0 * phase_range


def fractal_dimension_boxcount(mask: np.ndarray) -> float:
    binary = (mask > 0)
    if not binary.any(): return np.nan
    sizes, counts = [], []
    for size in [2, 4, 8, 16, 32]:
        h_b, w_b = binary.shape[0]//size, binary.shape[1]//size
        counts.append(1 if h_b==0 or w_b==0 else
                      int(binary[:h_b*size, :w_b*size].reshape(h_b,size,w_b,size).any(axis=(1,3)).sum()))
        sizes.append(size)
    return float(-np.polyfit(np.log(sizes), np.log(np.maximum(counts,1)), 1)[0])


def _safe_stat(arr: np.ndarray, prefix: str) -> dict:
    arr = arr[np.isfinite(arr)]
    keys = ['mean','median','std','skew','kurt','p10','p25','p75','p90','cv']
    if len(arr) < 3 or float(np.std(arr)) < 1e-6:
        return {f'{prefix}_{k}': np.nan for k in keys}
    mn = float(np.mean(arr))
    return {
        f'{prefix}_mean'  : mn,
        f'{prefix}_median': float(np.median(arr)),
        f'{prefix}_std'   : float(np.std(arr)),
        f'{prefix}_skew'  : float(stats.skew(arr)),
        f'{prefix}_kurt'  : float(stats.kurtosis(arr)),
        f'{prefix}_p10'   : float(np.percentile(arr, 10)),
        f'{prefix}_p25'   : float(np.percentile(arr, 25)),
        f'{prefix}_p75'   : float(np.percentile(arr, 75)),
        f'{prefix}_p90'   : float(np.percentile(arr, 90)),
        f'{prefix}_cv'    : float(np.std(arr)/mn) if mn != 0 else np.nan,
    }


def _glcm_features(gray_u8, mask_crop, prefix):
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm   = graycomatrix(gray_u8, distances=[1], angles=angles, levels=256, symmetric=True, normed=True)
    out    = {f'{prefix}_glcm_{p}': float(np.mean(graycoprops(glcm, p)))
              for p in ['contrast','correlation','energy','homogeneity','dissimilarity']}
    g = glcm.sum(axis=(2,3)); g /= g.sum()+1e-12
    out[f'{prefix}_glcm_entropy'] = float(-np.sum(g * np.log2(g + 1e-12)))
    sx   = cv2.Sobel(gray_u8, cv2.CV_64F, 1, 0, ksize=3)
    sy   = cv2.Sobel(gray_u8, cv2.CV_64F, 0, 1, ksize=3)
    roi  = np.sqrt(sx**2+sy**2)[mask_crop > 0]
    out[f'{prefix}_gradient_mean'] = float(roi.mean()) if roi.size else np.nan
    out[f'{prefix}_gradient_std']  = float(roi.std())  if roi.size else np.nan
    out[f'{prefix}_laplacian_var'] = float(cv2.Laplacian(gray_u8, cv2.CV_64F).var())
    return out


def _radial_features(img2d, mask2d, prefix, n_bands=4):
    nan_keys = [f'{prefix}_radial_band{i}' for i in range(n_bands)] + \
               [f'{prefix}_radial_gradient', f'{prefix}_radial_skew', f'{prefix}_nc_ratio_proxy']
    props = regionprops(mask2d.astype(np.uint8))
    if not props: return {k: np.nan for k in nan_keys}
    cy, cx   = props[0].centroid
    yy, xx   = np.indices(mask2d.shape)
    dist_map = np.sqrt((yy-cy)**2 + (xx-cx)**2)
    in_mask  = mask2d > 0
    max_r    = dist_map[in_mask].max() + 1e-6
    bm = []
    for i in range(n_bands):
        band = in_mask & (dist_map >= max_r*i/n_bands) & (dist_map < max_r*(i+1)/n_bands)
        bm.append(float(img2d[band].mean()) if band.any() else np.nan)
    out = {f'{prefix}_radial_band{i}': v for i, v in enumerate(bm)}
    out[f'{prefix}_radial_gradient'] = (bm[-1]-bm[0]) if not any(np.isnan(x) for x in [bm[0],bm[-1]]) else np.nan
    ri  = img2d[in_mask] * (dist_map[in_mask]/max_r)
    out[f'{prefix}_radial_skew']     = float(stats.skew(ri)) if len(ri) > 3 else np.nan
    iv  = img2d[in_mask & (dist_map < max_r*0.25)]
    ov  = img2d[in_mask & (dist_map >= max_r*0.25)]
    out[f'{prefix}_nc_ratio_proxy']  = float(iv.mean()/(ov.mean()+1e-6)) if iv.size and ov.size else np.nan
    return out


def _symmetry_score(mask2d):
    props = regionprops(mask2d.astype(np.uint8))
    if not props: return np.nan
    cy, cx = props[0].centroid; ori = props[0].orientation
    pad    = min(max(*mask2d.shape), 128)
    mp     = np.pad(mask2d.astype(np.float32), pad, mode='constant')
    M      = cv2.getRotationMatrix2D((float(cx+pad), float(cy+pad)), -float(np.degrees(ori)), 1.0)
    rot    = cv2.warpAffine(mp, M, (mp.shape[1], mp.shape[0]))
    rb     = (rot > 0.5).astype(np.uint8)
    fl     = np.flipud(rb)
    inter  = np.logical_and(rb, fl).sum()
    union  = np.logical_or(rb, fl).sum()
    return float(inter/union) if union > 0 else np.nan


def extract_instance_features(bf_gray, phase_rgb, mask_bin,
                               instance_id=0, source_file='', img_shape=None):
    """
    提取单实例特征。
    ★ AGG修复2/3：is_aggregate 判断使用修正后的阈值
         AREA_THRESHOLD_UM2=150, SOLIDITY_THRESHOLD=0.75, ASPECT_RATIO_THRESHOLD=3.0
    """
    px   = PIXEL_SIZE
    mask = (mask_bin > 0).astype(np.uint8)
    if mask.sum() < MIN_INSTANCE_PX:
        return None, None

    phase_rad = color_to_phase(phase_rgb)
    props     = regionprops(mask, intensity_image=bf_gray)
    if not props:
        return None, None
    rp = props[0]
    cy, cx = rp.centroid

    # 边缘过滤
    if img_shape is not None:
        H, W  = img_shape
        m     = EDGE_MARGIN_PX
        r0e, c0e, r1e, c1e = rp.bbox
        if cy < m or cy > H-m or cx < m or cx > W-m or r0e < m or c0e < m or r1e > H-m or c1e > W-m:
            return None, None

    area_px  = float(rp.area)
    area_um2 = area_px * px**2
    perim    = max(float(rp.perimeter), 1e-6)
    maj      = float(rp.major_axis_length)
    mno      = max(float(rp.minor_axis_length), 1e-6)
    solidity = float(rp.solidity)
    ax_ratio = maj / mno

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # ★ AGG修复2/3：is_aggregate 使用校准后的阈值
    #   原值：area>48 µm² OR solidity<0.85 OR axis_ratio>1.8
    #   修正值：area>150 µm² OR solidity<0.75 OR axis_ratio>3.0
    #
    #   逻辑说明：
    #   - 面积 >150 µm²：约等效直径 14 µm，明确超过激活单体血小板上限
    #   - solidity <0.75：形状高度非凸，有缺口或分叉，典型聚集体特征
    #   - axis_ratio >3.0：极端细长，通常是多个血小板线性排列
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    is_agg = (
        area_um2 > AREA_THRESHOLD_UM2 or
        solidity < SOLIDITY_THRESHOLD or
        ax_ratio > ASPECT_RATIO_THRESHOLD
    )

    bf_vals   = bf_gray[mask > 0]
    ph_vals   = phase_rad[mask > 0]
    alpha_si  = ALPHA * 1e-3
    dry_mass  = (WAVELENGTH / (2*np.pi*alpha_si)) * float(ph_vals.sum()) * PIXEL_SIZE**2 * 100

    r0, c0, r1, c1 = rp.bbox
    rec = {
        'instance_id'      : instance_id,
        'source_file'      : source_file,
        # 几何
        'area_px'          : area_px,
        'area_um2'         : area_um2,
        'perimeter'        : perim,
        'circularity'      : 4*np.pi*area_px/perim**2,
        'eccentricity'     : float(rp.eccentricity),
        'axis_ratio'       : ax_ratio,
        'solidity'         : solidity,
        'euler_number'     : float(rp.euler_number),
        'symmetry_score'   : _symmetry_score(mask),
        'fractal_dimension': fractal_dimension_boxcount(mask),
        # 聚集体标签
        'is_aggregate'     : int(is_agg),
        # 强度
        'bf_iod'           : float(bf_vals.sum() * px**2),
        'ph_iod'           : float(ph_vals.sum() * px**2),
        'ph_saturated'     : int(float(np.std(ph_vals)) < 1e-6),
        'dry_mass_pg'      : dry_mass,
        # bbox & 质心
        'bbox_minr': int(r0), 'bbox_minc': int(c0),
        'bbox_maxr': int(r1), 'bbox_maxc': int(c1),
        'centroid_y': float(cy), 'centroid_x': float(cx),
    }
    rec.update(_safe_stat(bf_vals,  'bf'))
    rec.update(_safe_stat(ph_vals,  'ph'))
    rec.update(_radial_features(bf_gray,   mask, 'bf'))
    rec.update(_radial_features(phase_rad, mask, 'ph'))

    bf_u8 = (bf_gray[r0:r1, c0:c1] * 255).astype(np.uint8)
    rec.update(_glcm_features(bf_u8, mask[r0:r1, c0:c1], 'bf'))

    return rec, rp.centroid


# 顶层 worker（多进程必须在模块级别）
def _extract_worker(args):
    bf_gray, phase_rgb, mask_bin, inst_id, source_file, img_shape = args
    return extract_instance_features(
        bf_gray, phase_rgb, mask_bin,
        instance_id=inst_id, source_file=source_file, img_shape=img_shape
    )


print('§6 特征提取工具就绪 ✓')

§6 特征提取工具就绪 ✓


# ║  §7  图合并聚集体算法                                                ║


In [22]:
# § 图合并聚集体算法已移除。
# 聚集体判定统一在特征提取（§6）和跨图邻居密度（§9）中完成。
print('§7 已移除（图合并算法不再使用）')

§7 已移除（图合并算法不再使用）


# ║  §8  加载模型                                                        ║


In [26]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ★ P1修复：bsize=256
eval_model = DualBranchTransformer(pretrained_path=None, bsize=BSIZE, dtype=torch.float32).to(device)

if not os.path.exists(BEST_CKPT_PATH):
    raise FileNotFoundError(f'best.pth 不存在: {BEST_CKPT_PATH}')
ck = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else \
     ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck

if isinstance(ck, dict) and 'model' in ck:
    print(f'✅ 加载成功  epoch={ck.get("epoch","?")}  val_loss={ck.get("best_loss",float("nan")):.6f}')
else:
    print('✅ 加载成功')

missing, unexpected = eval_model.load_state_dict(sd, strict=False)
fusion_m = [k for k in missing if 'fusion' in k]
other_m  = [k for k in missing if 'fusion' not in k]
print(f'   FusionBlock 缺失（正常）: {len(fusion_m)} keys')
if other_m:    print(f'   ⚠️  其他缺失: {other_m[:5]}')
if unexpected: print(f'   ⚠️  多余 keys: {unexpected[:5]}')

if device.type == 'cuda':
    eval_model = eval_model.half()
    print('   模型已转为 float16 (AMP 推理)')
eval_model.eval()
print('eval_model 就绪 ✓')

Device: cuda
✅ 加载成功  epoch=36  val_loss=0.162288
   FusionBlock 缺失（正常）: 0 keys
   模型已转为 float16 (AMP 推理)
eval_model 就绪 ✓


# ║  §9  主推理循环                                                      ║


In [32]:
# ╔══════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════╝
import time
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import cdist
from collections import defaultdict

eval_model.eval()
# BACKUP_CSV = os.path.join(CSV_OUT_DIR, 'BACKUP_latest.csv') # Moved to feJD96KMa2Zb

# ── 断点续跑 ────────────────────────────────────────────────
def _load_done():
    if not os.path.exists(BACKUP_CSV):
        print('  [续跑] 无备份，从头开始')
        return [], set()
    try:
        df   = pd.read_csv(BACKUP_CSV, low_memory=False)
        rec  = df.to_dict('records')
        done = set(df['unique_key'].dropna().unique()) if 'unique_key' in df.columns else set()
        print(f'  [续跑] 已加载 {len(rec)} 条，{len(done)} 张图已完成')
        return rec, done
    except Exception as e:
        print(f'  [警告] 读取备份失败: {e}')
        return [], set()

# Fix-A: 原代码调用了两次，第一次返回值被丢弃。现只调用一次。
existing, done_keys = _load_done()
all_records     = list(existing)
no_det_records  = []
group_centroids = defaultdict(list)

for gi, r in enumerate(existing):
    cy = r.get('centroid_y')
    cx = r.get('centroid_x')
    if cy is not None and cx is not None:
        group_centroids[(r.get('drug', 'unknown'), r.get('time_pt', 'unknown'))].append(
            (gi, (float(cy), float(cx)))
        )

todo = [s for s in all_samples if s['unique_key'] not in done_keys]
print(f'总图像: {len(all_samples)}  已完成: {len(done_keys)}  待处理: {len(todo)}')

_n_workers = FEAT_WORKERS if FEAT_WORKERS > 0 else min(os.cpu_count(), 8)

# ── 主循环 ────────────────────────────────────────────────
for img_idx, sample in enumerate(tqdm(todo, desc='Inference', unit='img')):
    t0         = time.time()
    image_name = sample['image_name']
    drug       = sample['drug']
    time_pt    = sample['time_pt']
    split      = sample['split']
    rel_dir    = sample['rel_dir']
    folder_id  = sample['folder_id']

    # 读取图像
    try:
        ph_rgb  = read_image_rgb(sample['phase'])
        bf_gray = read_image_gray(sample['intensity'])
        bf_3ch  = bf_to_3ch(bf_gray)
    except Exception as e:
        no_det_records.append({'image_name': image_name, 'drug': drug,
                               'time_pt': time_pt, 'reason': f'read:{e}'})
        continue

    img_shape = bf_gray.shape

    # GPU 推理
    try:
        logit_full, _, _ = sliding_window_inference(eval_model, ph_rgb, bf_3ch)
        pred_mask        = pred_to_mask(logit_full)
        n_instances      = int(pred_mask.max())
    except Exception as e:
        no_det_records.append({'image_name': image_name, 'drug': drug,
                               'time_pt': time_pt, 'reason': f'infer:{e}'})
        continue

    # 保存掩膜
    if SAVE_MASKS:
        mdir = os.path.join(MASK_OUT_DIR, split, rel_dir)
        os.makedirs(mdir, exist_ok=True)
        tifffile.imwrite(
            os.path.join(mdir, sample['stem'] + '_pred_mask.tiff'),
            pred_mask.astype(np.uint16)
        )

    if n_instances == 0:
        no_det_records.append({'image_name': image_name, 'drug': drug,
                               'time_pt': time_pt, 'reason': 'no_detection'})
        continue

    infer_time = time.time() - t0

    # 特征提取
    args = [
        (bf_gray, ph_rgb, (pred_mask == iid).astype(np.uint8), iid, image_name, img_shape)
        for iid in range(1, n_instances + 1)
    ]
    if _n_workers > 1 and n_instances >= 4:
        with ProcessPoolExecutor(max_workers=_n_workers) as pool:
            raw = list(pool.map(_extract_worker, args))
    else:
        raw = [_extract_worker(a) for a in args]

    valid = [(f, c) for f, c in raw if f is not None]
    if not valid:
        no_det_records.append({'image_name': image_name, 'drug': drug,
                               'time_pt': time_pt, 'reason': 'all_filtered'})
        continue

    meta = {
        'image_name'  : image_name,
        'split'       : split,
        'source_file' : image_name,
        'unique_key'  : sample['unique_key'],
        'drug'        : drug,
        'time_pt'     : time_pt,
        'rel_dir'     : rel_dir,
        'folder_id'   : folder_id,
        'infer_time_s': round(infer_time, 4),
    }

    # Fix-B: all_records 写入和 group_centroids 更新统一放在 if/else 外部。
    # 原代码这两段逻辑只在 else（非 graph_merge）分支里，导致主路径永远不写入记录。
    for feat, _ in valid:
        feat.update(meta)
        cy = feat.get('centroid_y')
        cx = feat.get('centroid_x')
        if cy is not None and cx is not None:
            group_centroids[(drug, time_pt)].append(
                (len(all_records), (float(cy), float(cx)))
            )
        all_records.append(feat)

    done_keys.add(sample['unique_key'])

    # 定期备份
    if (img_idx + 1) % SAVE_INTERVAL == 0:
        df_tmp = pd.DataFrame(all_records)
        # Fix-C: 原 fillna({'source_file': df_tmp['image_name']}) 将整列传入而非标量，且列缺失时直接崩溃。
        if 'source_file' in df_tmp.columns and 'image_name' in df_tmp.columns:
            df_tmp['source_file'] = df_tmp['source_file'].fillna(df_tmp['image_name'])
        df_tmp = df_tmp.drop_duplicates(subset=['unique_key', 'instance_id'], keep='last')
        df_tmp.to_csv(BACKUP_CSV, index=False)
        print(f'  [备份] {img_idx + 1} 张已处理，{len(df_tmp)} 条实例记录')

# Fix-D: 以下代码原本缩进在 for 循环内（每张图都重建 DataFrame），已正确移到循环外。

# ── 跨图邻居密度判定 ─────────────────────────────────────────────
print('跨图邻居密度判定...')
for (drug_g, tp_g), cent_list in group_centroids.items():
    if len(cent_list) < 2:
        continue
    idxs  = [x[0] for x in cent_list]
    cents = np.array([x[1] for x in cent_list])
    D     = cdist(cents, cents)
    np.fill_diagonal(D, np.inf)
    for li, gi in enumerate(idxs):
        if (D[li] < NEIGHBOR_DIST_THR).sum() > NEIGHBOR_COUNT_THR:
            all_records[gi]['is_aggregate'] = 1

# ── 最终去重 & 保存 ───────────────────────────────────────────
df_all = pd.DataFrame(all_records)
df_all = df_all.drop_duplicates(
    subset=['unique_key', 'instance_id'], keep='last'
).reset_index(drop=True)

# 列排序
front = ['image_name', 'unique_key', 'instance_id', 'split', 'drug', 'time_pt',
         'folder_id', 'rel_dir', 'is_aggregate', 'area_px', 'area_um2',
         'dry_mass_pg', 'centroid_y', 'centroid_x']
other = [c for c in df_all.columns if c not in front]
df_all = df_all[[c for c in front if c in df_all.columns] + other]

# 分 split x drug x time_pt 保存
for (sp, dr, tp), sub in df_all.groupby(['split', 'drug', 'time_pt']):
    sub.to_csv(os.path.join(CSV_OUT_DIR, f'{sp}_{dr}_{tp}_features.csv'), index=False)

df_all.to_csv(os.path.join(CSV_OUT_DIR, 'ALL_features.csv'), index=False)
pd.DataFrame(no_det_records).to_csv(
    os.path.join(CSV_OUT_DIR, 'no_detection_summary.csv'), index=False
)

# 聚集体占比统计
agg_stat = (
    df_all.groupby(['split', 'drug', 'time_pt'])['is_aggregate']
    .agg(total='count', n_agg='sum')
    .assign(agg_ratio=lambda x: x.n_agg / x.total * 100)
    .reset_index()
)
agg_stat.to_csv(os.path.join(CSV_OUT_DIR, 'aggregate_ratio_by_group.csv'), index=False)

print(f'\n[Done] 推理完成！')
print(f'  有效实例数  : {len(df_all)}')
print(f'  无检测图像  : {len(no_det_records)}')
print(f'  ALL_features: {os.path.join(CSV_OUT_DIR, "ALL_features.csv")}')

print('\n=== 聚集体占比（前20行）===')
print(agg_stat.head(20).to_string(index=False))

print('\n=== 各实例 is_aggregate 分布 ===')
print(df_all['is_aggregate'].value_counts().rename({0: '单体(0)', 1: '聚集体(1)'}))


  [续跑] 已加载 34730 条，33403 张图已完成
总图像: 45128  已完成: 33403  待处理: 11725


Inference:   0%|          | 0/11725 [00:00<?, ?img/s]

  [备份] 100 张已处理，34770 条实例记录
  [备份] 150 张已处理，34820 条实例记录
  [备份] 200 张已处理，34871 条实例记录
  [备份] 250 张已处理，34924 条实例记录
  [备份] 300 张已处理，34974 条实例记录
  [备份] 350 张已处理，35024 条实例记录
  [备份] 400 张已处理，35072 条实例记录
  [备份] 450 张已处理，35123 条实例记录
  [备份] 500 张已处理，35171 条实例记录
  [备份] 550 张已处理，35221 条实例记录
  [备份] 600 张已处理，35270 条实例记录
  [备份] 650 张已处理，35320 条实例记录
  [备份] 700 张已处理，35373 条实例记录
  [备份] 750 张已处理，35427 条实例记录
  [备份] 800 张已处理，35476 条实例记录
  [备份] 850 张已处理，35526 条实例记录
  [备份] 900 张已处理，35577 条实例记录
  [备份] 950 张已处理，35627 条实例记录
  [备份] 1000 张已处理，35678 条实例记录
  [备份] 1050 张已处理，35728 条实例记录
  [备份] 1100 张已处理，35780 条实例记录
  [备份] 1150 张已处理，35831 条实例记录
  [备份] 1200 张已处理，35881 条实例记录
  [备份] 1250 张已处理，35932 条实例记录
  [备份] 1300 张已处理，35982 条实例记录
  [备份] 1350 张已处理，36032 条实例记录
  [备份] 1400 张已处理，36082 条实例记录
  [备份] 1450 张已处理，36132 条实例记录
  [备份] 1500 张已处理，36181 条实例记录
  [备份] 1550 张已处理，36231 条实例记录
  [备份] 1600 张已处理，36280 条实例记录
  [备份] 1650 张已处理，36332 条实例记录
  [备份] 1700 张已处理，36385 条实例记录
  [备份] 1750 张已处理，36441 条实例记录
  [备份] 1800 张已处理，36491 条实例记录
  [

/tmp/ipykernel_449/3097201355.py:36: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_skew'  : float(stats.skew(arr)),
/tmp/ipykernel_449/3097201355.py:37: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_kurt'  : float(stats.kurtosis(arr)),


  [备份] 7300 张已处理，42210 条实例记录
  [备份] 7350 张已处理，42260 条实例记录
  [备份] 7400 张已处理，42309 条实例记录
  [备份] 7450 张已处理，42360 条实例记录
  [备份] 7500 张已处理，42412 条实例记录
  [备份] 7550 张已处理，42465 条实例记录
  [备份] 7600 张已处理，42516 条实例记录
  [备份] 7650 张已处理，42568 条实例记录
  [备份] 7700 张已处理，42618 条实例记录
  [备份] 7750 张已处理，42672 条实例记录
  [备份] 7800 张已处理，42723 条实例记录
  [备份] 7850 张已处理，42775 条实例记录
  [备份] 7900 张已处理，42829 条实例记录
  [备份] 7950 张已处理，42879 条实例记录
  [备份] 8000 张已处理，42930 条实例记录
  [备份] 8050 张已处理，42981 条实例记录
  [备份] 8100 张已处理，43033 条实例记录
  [备份] 8150 张已处理，43086 条实例记录
  [备份] 8200 张已处理，43137 条实例记录
  [备份] 8250 张已处理，43190 条实例记录
  [备份] 8300 张已处理，43242 条实例记录
  [备份] 8350 张已处理，43297 条实例记录
  [备份] 8400 张已处理，43348 条实例记录
  [备份] 8450 张已处理，43400 条实例记录
  [备份] 8500 张已处理，43454 条实例记录
  [备份] 8550 张已处理，43507 条实例记录
  [备份] 8600 张已处理，43559 条实例记录
  [备份] 8650 张已处理，43610 条实例记录
  [备份] 8700 张已处理，43661 条实例记录
  [备份] 8750 张已处理，43713 条实例记录
  [备份] 8800 张已处理，43764 条实例记录
  [备份] 8850 张已处理，43814 条实例记录
  [备份] 8900 张已处理，43864 条实例记录
  [备份] 8950 张已处理，43915 条实例记录
  [备份] 9000 张已

In [28]:
import pandas as pd

# 1. 读取 CSV 文件
# 修复：pd.read_csv 期望单个文件路径，而不是目录和文件名分开
df = pd.read_csv(BACKUP_CSV)

# 2. 计算占比
# normalize=True 会直接返回比例
proportion = df['is_aggregate'].value_counts(normalize=True)

# 3. 如果想以百分比形式展示：
percentage = df['is_aggregate'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%'

print("比例分布：")
print(proportion)
print("\n百分比分布：")
print(percentage)

NameError: name 'BACKUP_CSV' is not defined

# ║  §10  单张图像可视化验证                                             ║


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ╚══════════════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEMO_IDX = 0   # 修改为任意样本索引

if DEMO_IDX < len(all_samples):
    sample = all_samples[DEMO_IDX]
    print(f'样本: {sample["image_name"]}')
    print(f'Split={sample["split"]}  Drug={sample["drug"]}  Time={sample["time_pt"]}')

    ph_d  = read_image_rgb(sample['phase'])
    bf_d  = read_image_gray(sample['intensity'])
    bf3_d = bf_to_3ch(bf_d)

    eval_model.eval()
    logit_d, _, _ = sliding_window_inference(eval_model, ph_d, bf3_d)
    mask_d        = pred_to_mask(logit_d)
    cellprob_d    = sigmoid(logit_d[0])

    # 快速特征提取（仅用于显示聚集体标签）
    colors = {}
    np.random.seed(42)
    canvas = (bf_d * 255).clip(0,255).astype(np.uint8)
    canvas = np.stack([canvas]*3, axis=-1)
    for iid in np.unique(mask_d)[1:]:
        m_bin = (mask_d == iid).astype(np.uint8)
        feat, _ = extract_instance_features(bf_d, ph_d, m_bin,
                                             instance_id=iid, img_shape=bf_d.shape)
        is_agg = feat.get('is_aggregate', 0) if feat else 0
        col    = np.array([220, 60, 60]) if is_agg else np.array([60, 180, 60])
        colors[iid] = ('agg' if is_agg else 'single', col)
        for c in range(3):
            canvas[mask_d==iid, c] = (0.55*col[c] + 0.45*canvas[mask_d==iid,c]).astype(np.uint8)

    n_single = sum(1 for v in colors.values() if v[0]=='single')
    n_agg    = sum(1 for v in colors.values() if v[0]=='agg')

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(ph_d);                                axes[0].set_title('Phase (RGB)')
    axes[1].imshow(bf_d, cmap='gray');                   axes[1].set_title('Brightfield')
    axes[2].imshow(cellprob_d, cmap='hot', vmin=0, vmax=1); axes[2].set_title(f'Cell prob (thr={PROB_THRESHOLD})')
    axes[3].imshow(canvas)
    axes[3].set_title(f'Mask: {n_single} 单体(绿) + {n_agg} 聚集体(红)')
    patches = [mpatches.Patch(color=[0.24,0.71,0.24], label=f'单体 {n_single}'),
               mpatches.Patch(color=[0.86,0.24,0.24], label=f'聚集体 {n_agg}')]
    axes[3].legend(handles=patches, loc='upper right', fontsize=9)
    for ax in axes: ax.axis('off')
    plt.suptitle(f'{sample["image_name"]} | {sample["drug"]} {sample["time_pt"]}', fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f'\n检测实例总数: {mask_d.max()}')
    print(f'单体: {n_single}  聚集体: {n_agg}  聚集体占比: {n_agg/max(mask_d.max(),1)*100:.1f}%')
    print(f'\n面积阈值: {AREA_THRESHOLD_UM2} µm²  Solidity阈值: {SOLIDITY_THRESHOLD}  长宽比阈值: {ASPECT_RATIO_THRESHOLD}')